# Baseline Models

This notebook trains and evaluates three baseline models (Logistic Regression, Random Forest, and a simple Neural Network) on GUIDE-seq data. Performance is evaluated on the gRNA-disjoint GUIDE-seq test set and cross-dataset on CHANGE-seq to establish reference performance.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import os

from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

In [2]:
# Loads preprocessed one-hot arrays
DATA_DIR = 'data/processed/'

X_guide_train = np.load(DATA_DIR + 'X_guide_train.npy')
X_guide_test = np.load(DATA_DIR + 'X_guide_test.npy')
X_change_test = np.load(DATA_DIR + 'X_change_test.npy')

y_guide_train = np.load(DATA_DIR + 'y_guide_train.npy')
y_guide_test = np.load(DATA_DIR + 'y_guide_test.npy')
y_change_test = np.load(DATA_DIR + 'y_change_test.npy')

# Checks feature/label alignment
for name, X, y in [
    ('GUIDE train', X_guide_train, y_guide_train),
    ('GUIDE test', X_guide_test, y_guide_test),
    ('CHANGE test', X_change_test, y_change_test)
]:
    assert X.shape[0] == len(y)
    assert X.shape[1] == 184
    print(
        f'{name}: X={X.shape}, y={y.shape}, '
        f'positives={int(y.sum()):,}'
    )

GUIDE train: X=(1229148, 184), y=(1229148,), positives=1,371
GUIDE test: X=(248614, 184), y=(248614,), positives=90
CHANGE test: X=(520991, 184), y=(520991,), positives=6,610


### Logistic Regression

In [3]:
# Training on GUIDE-seq and evaluating within-dataset and cross-dataset
# Using class_weight='balanced' to handle the severe class imbalance (~99.9% negatives)
lr_guide = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=2000)
lr_guide.fit(X_guide_train, y_guide_train)

# In-distribution evaluation
y_guide_proba_lr = lr_guide.predict_proba(X_guide_test)[:, 1]
print('GUIDE-seq (in-distribution):')
print(f'  AUROC: {roc_auc_score(y_guide_test, y_guide_proba_lr):.4f}')
print(f'  AUPRC: {average_precision_score(y_guide_test, y_guide_proba_lr):.4f}')

# OOD evaluation on CHANGE-seq
y_change_proba_lr = lr_guide.predict_proba(X_change_test)[:, 1]
print('\nCHANGE-seq (out-of-distribution):')
print(f'  AUROC: {roc_auc_score(y_change_test, y_change_proba_lr):.4f}')
print(f'  AUPRC: {average_precision_score(y_change_test, y_change_proba_lr):.4f}')

GUIDE-seq (in-distribution):
  AUROC: 0.6471
  AUPRC: 0.0005

CHANGE-seq (out-of-distribution):
  AUROC: 0.8612
  AUPRC: 0.0840


### Random Forest

In [4]:
# Random Forest with balanced class weights
rf_guide = RandomForestClassifier(
    n_estimators=100, 
    class_weight='balanced', 
    random_state=2000, 
    n_jobs=-1
)
rf_guide.fit(X_guide_train, y_guide_train)

# In-distribution evaluation
y_guide_proba_rf = rf_guide.predict_proba(X_guide_test)[:, 1]
print('GUIDE-seq (in-distribution):')
print(f'  AUROC: {roc_auc_score(y_guide_test, y_guide_proba_rf):.4f}')
print(f'  AUPRC: {average_precision_score(y_guide_test, y_guide_proba_rf):.4f}')

# OOD evaluation
y_change_proba_rf = rf_guide.predict_proba(X_change_test)[:, 1]
print('\nCHANGE-seq (out-of-distribution):')
print(f'  AUROC: {roc_auc_score(y_change_test, y_change_proba_rf):.4f}')
print(f'  AUPRC: {average_precision_score(y_change_test, y_change_proba_rf):.4f}')

GUIDE-seq (in-distribution):
  AUROC: 0.4954
  AUPRC: 0.0004

CHANGE-seq (out-of-distribution):
  AUROC: 0.5576
  AUPRC: 0.0461


### Simple Neural Network

In [5]:
# Simple feedforward network with two hidden layers
class SimpleNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.network(x)

# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Calculating pos_weight to handle class imbalance 
num_pos = y_guide_train.sum()
num_neg = len(y_guide_train) - num_pos
pos_weight = torch.tensor(
    [num_neg / num_pos],
    dtype=torch.float32,
    device=device
)
print(f'pos_weight: {pos_weight.item():.1f}')

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

Using device: cuda
pos_weight: 895.5


In [6]:
# Reproducibility
torch.manual_seed(2000)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(2000)

# Converts numpy arrays to PyTorch tensors and creates DataLoader
X_train_tensor = torch.from_numpy(X_guide_train).float()
y_train_tensor = torch.from_numpy(y_guide_train).float().unsqueeze(1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=512,
    shuffle=True
)

# Initialises model and optimiser
model = SimpleNN(
    input_dim=X_guide_train.shape[1]
).to(device)

optimiser = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

# Training loop
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimiser.zero_grad()

        preds = model(X_batch)
        loss = criterion(preds, y_batch)

        loss.backward()
        optimiser.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f'Epoch {epoch + 1:02d}/{EPOCHS} - '
        f'Loss: {avg_loss:.4f}'
    )

Epoch 01/10 - Loss: 0.8951
Epoch 02/10 - Loss: 0.7345
Epoch 03/10 - Loss: 0.6671
Epoch 04/10 - Loss: 0.6063
Epoch 05/10 - Loss: 0.5485
Epoch 06/10 - Loss: 0.4952
Epoch 07/10 - Loss: 0.4910
Epoch 08/10 - Loss: 0.4314
Epoch 09/10 - Loss: 0.4000
Epoch 10/10 - Loss: 0.3606


In [7]:
# Evaluating on GUIDE-seq test set (in-distribution)
model.eval()
with torch.no_grad():
    X_guide_test_tensor = torch.from_numpy(X_guide_test).float().to(device)
    logits = model(X_guide_test_tensor)
    y_guide_proba_nn = torch.sigmoid(logits).cpu().numpy().flatten()

print('GUIDE-seq (in-distribution):')
print(f'  AUROC: {roc_auc_score(y_guide_test, y_guide_proba_nn):.4f}')
print(f'  AUPRC: {average_precision_score(y_guide_test, y_guide_proba_nn):.4f}')

# OOD evaluation on CHANGE-seq
with torch.no_grad():
    X_change_test_tensor = torch.from_numpy(X_change_test).float().to(device)
    logits_ood = model(X_change_test_tensor)
    y_change_proba_nn = torch.sigmoid(logits_ood).cpu().numpy().flatten()

print('\nCHANGE-seq (out-of-distribution):')
print(f'  AUROC: {roc_auc_score(y_change_test, y_change_proba_nn):.4f}')
print(f'  AUPRC: {average_precision_score(y_change_test, y_change_proba_nn):.4f}')

GUIDE-seq (in-distribution):
  AUROC: 0.6818
  AUPRC: 0.0007

CHANGE-seq (out-of-distribution):
  AUROC: 0.7850
  AUPRC: 0.1115


### Baseline Results Summary

In [8]:
# Summary table of baseline performance
results = {
    'Model': ['Logistic Regression', 'Random Forest', 'Neural Network'],
    'GUIDE AUROC': [
        roc_auc_score(y_guide_test, y_guide_proba_lr),
        roc_auc_score(y_guide_test, y_guide_proba_rf),
        roc_auc_score(y_guide_test, y_guide_proba_nn)
    ],
    'GUIDE AUPRC': [
        average_precision_score(y_guide_test, y_guide_proba_lr),
        average_precision_score(y_guide_test, y_guide_proba_rf),
        average_precision_score(y_guide_test, y_guide_proba_nn)
    ],
    'CHANGE AUROC': [
        roc_auc_score(y_change_test, y_change_proba_lr),
        roc_auc_score(y_change_test, y_change_proba_rf),
        roc_auc_score(y_change_test, y_change_proba_nn)
    ],
    'CHANGE AUPRC': [
        average_precision_score(y_change_test, y_change_proba_lr),
        average_precision_score(y_change_test, y_change_proba_rf),
        average_precision_score(y_change_test, y_change_proba_nn)
    ]
}

results_df = pd.DataFrame(results)
results_df.round(4)

,Model,GUIDE AUROC,GUIDE AUPRC,CHANGE AUROC,CHANGE AUPRC
0,Logistic Regression,0.6471,0.0005,0.8612,0.0840
1,Random Forest,0.4954,0.0004,0.5576,0.0461
2,Neural Network,0.6818,0.0007,0.7850,0.1115


In [9]:
RESULTS_DIR = 'results/baselines/'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Save per-sample baseline predictions
np.save(RESULTS_DIR + 'guide_lr.npy', y_guide_proba_lr)
np.save(RESULTS_DIR + 'guide_rf.npy', y_guide_proba_rf)
np.save(RESULTS_DIR + 'guide_nn.npy', y_guide_proba_nn)

np.save(RESULTS_DIR + 'change_lr.npy', y_change_proba_lr)
np.save(RESULTS_DIR + 'change_rf.npy', y_change_proba_rf)
np.save(RESULTS_DIR + 'change_nn.npy', y_change_proba_nn)

# Save headline metrics
results_df.to_csv(
    RESULTS_DIR + 'baseline_metrics.csv',
    index=False
)

print('Baseline results saved to', RESULTS_DIR)

Baseline results saved to results/baselines/


### Overview

* Logistic Regression and the Neural Network provide stronger cross-dataset discrimination than the Random Forest in this baseline setup.
* AUPRC is strongly affected by class prevalence, so GUIDE-seq and CHANGE-seq AUPRC values should be interpreted relative to their respective positive rates rather than compared directly.
* Class weighting and `pos_weight` are used to address severe class imbalance during training. As a result, the raw predicted probabilities are not assumed to be calibrated; calibration is assessed separately in the evaluation analysis.